In [1]:
# Create an API Client
from anthropic import Anthropic
from pydantic import BaseModel

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Using output_config

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    message = client.messages.create(output_config=
        {
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "array"},
                        "detail_type": {"type": "array"},
                        "detail": {
                            "type": "object",
                            "properties": {
                                "state": {"type": "array"}
                            },
                            "additionalProperties": False
                        }
                    },
                    "required": ["source", "detail_type", "detail"],
                    "additionalProperties": False,                   
                },
                
            },
        },
        **params)

    return message.content[0].text

In [15]:
# Using Pydantic
class detailObject(BaseModel):
    state: list[str]

class bridgeRule(BaseModel):
    source: list[str]
    detail_type: list[str]
    detail: detailObject


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "output_format": bridgeRule
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    message = client.messages.parse(**params)

    return message.content[0].text

In [2]:
# using stop_sequences

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    message = client.messages.create(**params)

    return message.content[0].text

In [42]:
messages = []



add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=["5", "3, 4"])
print(answer)

{"source": ["custom.count"], "detail_type": ["count.completed"], "detail": {"state": ["1", "2", "3", "4", "


In [ ]:
# Using output_config/output_format

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
#add_assistant_message(messages, "```json")

text = chat(messages)
text

'{"source":["aws.ec2"],"detail_type":["EC2 Instance State-change Notification"],"detail":{"state":["running"]}}'

In [47]:
# Using stop_sequences

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [48]:
import json

json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [18]:
# Excercise

messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\n\naws ec2 describe-instances --region us-east-1\n\naws lambda list-functions'

In [19]:
from IPython.display import Markdown

Markdown(text)


aws s3 ls

aws ec2 describe-instances --region us-east-1

aws lambda list-functions
